# Task 3 — Insights & Recommendations

Self-contained: reloads the cleaned dataset independently. Every insight below is generated from a computed value in the cell above it — not a fixed sentence written in advance.

## 1. Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import os

if os.path.exists('ecommerce_orders_clean.csv'):
    df = pd.read_csv('ecommerce_orders_clean.csv', parse_dates=['order_date'])
else:
    import pandas as pd
    import numpy as np

    # ---------------------------------------------------------------
    # Self-contained data generation
    # This block is identical across all three notebooks so each one
    # can run independently (no shared kernel state required).
    # Seeded for reproducibility.
    # ---------------------------------------------------------------
    np.random.seed(42)

    n = 1200
    categories = ['Electronics','Clothing','Home & Kitchen','Books','Beauty','Sports','Toys','Grocery']
    payment_methods = ['Credit Card','Debit Card','Cash on Delivery','Digital Wallet']
    regions = ['North','South','East','West','Central']
    statuses = ['Delivered','Cancelled','Returned','Pending']
    status_weights = [0.62, 0.16, 0.14, 0.08]

    start_date = pd.Timestamp('2023-01-01')
    end_date = pd.Timestamp('2025-06-30')
    date_range_days = (end_date - start_date).days

    df = pd.DataFrame({
        'order_id': [f'ORD{100000+i}' for i in range(n)],
        'customer_id': np.random.randint(1000, 1500, size=n),
        'order_date': [start_date + pd.Timedelta(days=int(x)) for x in np.random.randint(0, date_range_days, size=n)],
        'product_category': np.random.choice(categories, size=n),
        'quantity': np.random.randint(1, 6, size=n),
        'unit_price': np.round(np.random.gamma(3, 25, size=n), 2),
        'discount_percent': np.random.choice([0,5,10,15,20,25], size=n, p=[0.35,0.2,0.2,0.15,0.07,0.03]),
        'payment_method': np.random.choice(payment_methods, size=n),
        'shipping_cost': np.round(np.random.uniform(2,15,size=n),2),
        'delivery_days': np.random.randint(1,15,size=n),
        'customer_rating': np.random.choice([1,2,3,4,5], size=n, p=[0.05,0.08,0.17,0.35,0.35]),
        'order_status': np.random.choice(statuses, size=n, p=status_weights),
        'region': np.random.choice(regions, size=n),
    })

    df['total_amount'] = np.round(
        df['quantity'] * df['unit_price'] * (1 - df['discount_percent']/100) + df['shipping_cost'], 2
    )

    # Inject realistic data-quality issues so the pipeline has something genuine to catch
    missing_idx = np.random.choice(df.index, size=25, replace=False)
    df.loc[missing_idx, 'customer_rating'] = np.nan
    df = pd.concat([df, df.sample(8, random_state=1)], ignore_index=True)

    df = df.drop_duplicates().copy()
    df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())

print(f'{df.shape[0]} rows ready for insight generation.')

## 2. Data-Driven Insights

In [ ]:
status_pct = df['order_status'].value_counts(normalize=True) * 100
cancel_return_rate = status_pct.get('Cancelled', 0) + status_pct.get('Returned', 0)

print(f"INSIGHT 1 — Order fulfillment risk")
print(f"{cancel_return_rate:.1f}% of all orders end in cancellation or return "
      f"({status_pct.get('Cancelled',0):.1f}% cancelled, {status_pct.get('Returned',0):.1f}% returned).")
print(f"That means roughly 1 in {round(100/cancel_return_rate)} orders never completes successfully.")

In [ ]:
top_category = df.groupby('product_category')['total_amount'].sum().idxmax()
top_category_rev = df.groupby('product_category')['total_amount'].sum().max()
rev_share = top_category_rev / df['total_amount'].sum() * 100

print(f"INSIGHT 2 — Revenue concentration")
print(f"'{top_category}' is the top revenue category, generating {rev_share:.1f}% of total revenue "
      f"(${top_category_rev:,.2f}).")

In [ ]:
corr_price = df[['unit_price','total_amount']].corr().iloc[0,1]
corr_qty = df[['quantity','total_amount']].corr().iloc[0,1]

print(f"INSIGHT 3 — What actually drives order value")
print(f"unit_price correlates with total_amount at r={corr_price:.2f}; quantity at r={corr_qty:.2f}.")
stronger = 'unit price' if abs(corr_price) > abs(corr_qty) else 'quantity'
print(f"'{stronger}' is the stronger driver of order value — pricing strategy matters more than upsell volume here.")

In [ ]:
cancel_return = df[df['order_status'].isin(['Cancelled','Returned'])]
rate_by_payment = (cancel_return.groupby('payment_method').size() / df.groupby('payment_method').size() * 100)
worst_payment = rate_by_payment.idxmax()
best_payment = rate_by_payment.idxmin()

print(f"INSIGHT 4 — Payment method risk")
print(f"'{worst_payment}' has the highest cancel/return rate at {rate_by_payment.max():.1f}%, "
      f"vs {rate_by_payment.min():.1f}% for '{best_payment}'.")
print(f"Gap: {rate_by_payment.max()-rate_by_payment.min():.1f} percentage points.")

In [ ]:
q1, q3 = df['total_amount'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = df[(df['total_amount'] < low) | (df['total_amount'] > high)]

print(f"INSIGHT 5 — High-value order concentration")
print(f"{len(outliers)} orders ({len(outliers)/len(df)*100:.1f}%) are statistical outliers on total_amount "
      f"(above ${high:,.2f}). These deserve separate handling in any revenue forecasting model "
      f"since they will otherwise skew the mean.")

## 3. Recommendations

Generated directly from the insights above:

1. **Investigate cancellations/returns by payment method first** — the gap between the best and worst payment method (see Insight 4) is a concrete, actionable lead, unlike a dataset-wide rate alone.
2. **Prioritize pricing review over upsell campaigns** if unit_price is confirmed as the stronger revenue driver (Insight 3) — discount and bundling strategy will move revenue more than pushing quantity.
3. **Model high-value orders separately** (Insight 5) — mixing outlier orders into a single average will bias any forecast or KPI toward a small number of large transactions.
4. **Protect the top revenue category** (Insight 2) with dedicated inventory and delivery SLAs, since a disruption there has an outsized effect on total revenue.

## 4. Limitations

- Dataset is synthetic; relationships are realistic in shape but not sourced from real transactions.
- Correlation values quantify linear relationships only — non-linear drivers of order value would need a separate model (e.g. tree-based feature importance) to detect.
- Cancellation/return reasons are not captured in the data, so Insight 4 identifies *where* risk concentrates, not *why*.